In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score


# load cleaned datasets
train_df = pd.read_csv("data/booking_reviews_train.csv")
test_df = pd.read_csv("data/booking_reviews_test.csv")

# features + labels
X_train = train_df[["text_combined", "cleanliness_score"]].copy()
X_test = test_df[["text_combined", "cleanliness_score"]].copy()

X_train["cleanliness_score"] = X_train["cleanliness_score"].fillna(0)
X_test["cleanliness_score"] = X_test["cleanliness_score"].fillna(0)

y_train = train_df["is_positive"]
y_test = test_df["is_positive"]

# model pipeline
preprocessor = ColumnTransformer([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english"
    ), "text_combined"),
    ("numeric", "passthrough", ["cleanliness_score"])
])

model = Pipeline([
    ("features", preprocessor),
    ("logreg", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# train + predict
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# evaluation
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))

print("\nReport:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

y_proba = model.predict_proba(X_test)[:, 1]
print("ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.845
Precision: 0.953
Recall: 0.83
F1 Score: 0.887

Report:
              precision    recall  f1-score   support

           0       0.65      0.89      0.75      1396
           1       0.95      0.83      0.89      3857

    accuracy                           0.85      5253
   macro avg       0.80      0.86      0.82      5253
weighted avg       0.87      0.85      0.85      5253

Confusion Matrix:
[[1237  159]
 [ 655 3202]]
ROC AUC Score: 0.933
